# Olist E-Commerce Analysis

Exploratory data analysis of Olist, a Brazilian e-commerce platform. This notebook answers three key business questions using a real dataset of over 99,000 orders placed between 2016 and 2018.

## 1. Data Loading & Exploration

Loading and exploring the main orders dataset to understand its structure, data types, and data quality.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/olist_orders_dataset.csv")
df.head()

In [ ]:
filas, n_columnas = df.shape
print(f'Hay {filas} filas y {n_columnas} columnas')

In [ ]:
nombres_columnas = list(df.columns)
print(f'Los nombres de las columnas son:\n {nombres_columnas}')

In [ ]:
df.info()

In [ ]:
df.value_counts("order_status")

In [ ]:
pedidos = df.shape[0]
entregados = df[df['order_status'] == 'delivered'].shape[0]
tasa_entrega = (entregados / pedidos) * 100
print(f'El porcentaje de pedidos entregados es: {tasa_entrega:.2f}%')

## 2. Revenue Analysis

Loading the order items dataset and calculating total revenue for delivered orders only. We then analyze how revenue evolved month by month between 2016 and 2018.

In [ ]:
df_items = pd.read_csv("../data/olist_order_items_dataset.csv")
df_items.head()

In [ ]:
filas, n_columnas = df_items.shape
print(f'Hay {filas} filas y {n_columnas} columnas')

In [ ]:
df_items.info()

In [ ]:
total_precio_ventas = df_items['price'].sum()
print(f'El precio total de las ventas es: R$ {total_precio_ventas:,.2f}')

In [ ]:
df_merged = df.merge(df_items, on='order_id')
df_merged.head()

In [ ]:
filas, n_columnas = df_merged.shape
print(f'Hay {filas} filas y {n_columnas} columnas')

df_delivered = df_merged[df_merged['order_status'] == 'delivered'].copy()
filas, n_columnas = df_delivered.shape
print(f'Pedidos entregados: {filas} filas y {n_columnas} columnas')

In [ ]:
total_precio_ventas_delivered = df_delivered['price'].sum()
print(f'Revenue total (solo entregados): R$ {total_precio_ventas_delivered:,.2f}')
print(f'Revenue no recuperado:           R$ {total_precio_ventas - total_precio_ventas_delivered:,.2f}')

In [ ]:
df_delivered['order_purchase_timestamp'] = pd.to_datetime(df_delivered['order_purchase_timestamp'])
df_delivered.info()

In [ ]:
df_delivered['mes'] = df_delivered['order_purchase_timestamp'].dt.to_period('M')
df_delivered.head()

In [ ]:
revenue_mensual = df_delivered.groupby('mes')['price'].sum()
print(revenue_mensual)
print(f'\nEl pico mas alto del revenue fue en: {revenue_mensual.idxmax()} debido al Black Friday')

In [ ]:
import matplotlib.pyplot as plt

# tamaño del gráfico, titulo, eje x, eje y, rotacion de fechas y ajuste de margenes automatico
plt.figure(figsize=(12, 5))
revenue_mensual.plot()
plt.title('Revenue Mensual - Olist 2016-2018')
plt.xlabel('Mes')
plt.ylabel('Revenue (R$)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Top Product Categories by Revenue

Loading the products dataset and joining it with delivered orders to identify which product categories generate the most revenue.

In [ ]:
df_products = pd.read_csv("../data/olist_products_dataset.csv")
df_products.head()

In [ ]:
filas, n_columnas = df_products.shape
print(f'Hay {filas} filas y {n_columnas} columnas')

In [ ]:
df_products.info()

In [ ]:
df_full = df_products.merge(df_delivered, on='product_id')
print(f'Shape del merge: {df_full.shape}')

In [ ]:
revenue_categoria = df_full.groupby('product_category_name')['price'].sum()
top10_categorias = revenue_categoria.sort_values(ascending=False).head(10)
print(top10_categorias)

In [ ]:
df_traduccion = pd.read_csv("../data/product_category_name_translation.csv")
top10_df = top10_categorias.reset_index()
top10_translated = top10_df.merge(df_traduccion, on='product_category_name')
print(top10_translated)

In [ ]:
top10_translated.plot(kind='barh', x='product_category_name_english', y='price', figsize=(10, 6))
plt.title('Top 10 Product Categories by Revenue - Olist')
plt.xlabel('Revenue (R$)')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

## 4. Revenue by State

Joining the customers dataset to analyze the geographic distribution of revenue across Brazilian states.

In [ ]:
df_customers = pd.read_csv("../data/olist_customers_dataset.csv")
df_customers.head()

In [ ]:
filas, n_columnas = df_customers.shape
print(f'Hay {filas} filas y {n_columnas} columnas')
df_customers.info()

In [ ]:
df_merged_customers = df_customers.merge(df_full, on='customer_id')
filas, n_columnas = df_merged_customers.shape
print(f'Hay {filas} filas y {n_columnas} columnas')

In [ ]:
revenue_customers = df_merged_customers.groupby('customer_state')['price'].sum()
revenue_customers = revenue_customers.sort_values(ascending=False).head(10)
print(revenue_customers)

In [ ]:
revenue_customers.plot(kind='barh', figsize=(10, 6))
plt.title('Top 10 States by Revenue - Olist')
plt.xlabel('Revenue (R$)')
plt.ylabel('States')
plt.tight_layout()
plt.show()

## Conclusions

- **97.02% delivery rate**: Olist successfully delivered 96,478 out of 99,441 orders. Undelivered orders represent R$ 370,145 in unrecovered revenue.
- **Black Friday peak**: November 2017 was the highest-revenue month (R$ 987,765), nearly doubling the average monthly revenue.
- **Top category**: Health & Beauty leads with R$ 1,233,131 in revenue, followed by Watches & Gifts and Bed, Bath & Table.
- **Geographic concentration**: São Paulo (SP) dominates with R$ 5,067,633 — almost 3x the revenue of second-place Rio de Janeiro (RJ).